# Class 2 — Tokens, Temperature & Context Windows
**Week 2: Introduction to LLMs — "The Black Box / Neural Signal"**

### Learning objectives
By the end of this notebook you will be able to:
- Count real tokens with `tiktoken` and see why token count isn't word count
- Compare low- vs. high-temperature output from a real LLM call
- Estimate the dollar cost of an API request from its token count
- Simulate trimming a conversation to fit inside a small context window

Section 1 works with no API key. Sections 2-3 call a real model via Groq and need a `GROQ_API_KEY` to produce live output — see Setup.

## Setup
**Running in Google Colab:**
1. Get a free key from https://console.groq.com/keys
2. Click the key icon (🔑 Secrets) in the left sidebar
3. Add a secret named `GROQ_API_KEY`, paste your key, toggle **Notebook access** on
4. Run the two setup cells below

**Elsewhere:** set `GROQ_API_KEY` as an environment variable before launching Jupyter.

In [ ]:
!pip install -q groq tiktoken

In [ ]:
import os

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    print(
        "No API key found — Section 1 still works.\n"
        "In Colab: add a secret named GROQ_API_KEY via the 🔑 Secrets panel and enable notebook access.\n"
        "Elsewhere: set GROQ_API_KEY as an environment variable before launching Jupyter."
    )
else:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    print("GROQ_API_KEY loaded — live sections will work.")

## 1. Counting Real Tokens
Models don't see words — they see tokens, sub-word chunks produced by a tokenizer. `tiktoken` is OpenAI's tokenizer library; Groq's Llama models use a different tokenizer under the hood, but `cl100k_base` is close enough to build intuition, and needs no API call.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

examples = [
    "Generative AI is transformative.",
    "supercalifragilisticexpialidocious",
    "La inteligencia artificial generativa es transformadora.",
]

for text in examples:
    tokens = enc.encode(text)
    print(f"{len(tokens):>3} tokens — {text!r}")
    print(f"     pieces: {[enc.decode([t]) for t in tokens]}\n")

Notice the Spanish sentence uses noticeably more tokens for a similar-length idea — tokenizers are typically trained mostly on English text, so other languages split into smaller, less efficient pieces.

## 2. Same Prompt, Two Temperatures (Live)
This cell sends one prompt to Groq twice: once at `temperature=0.1`, once at `temperature=1.2`. Needs `GROQ_API_KEY`.

In [ ]:
def call_llm(prompt, temperature=0.7, model="llama-3.3-70b-versatile", max_tokens=80):
    """Send a single-turn prompt to Groq at a given temperature and return the text, or an error message."""
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        return "Error: GROQ_API_KEY is not set."
    try:
        from groq import Groq
        client = Groq(api_key=api_key)
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error calling Groq: {e}"

prompt = "Finish this sentence: The old lighthouse keeper opened the door and saw"

if os.environ.get("GROQ_API_KEY"):
    print("temperature = 0.1:\n", call_llm(prompt, temperature=0.1))
    print("\ntemperature = 1.2:\n", call_llm(prompt, temperature=1.2))
else:
    print("Set GROQ_API_KEY, then re-run this cell to see live output at both temperatures.")

## 3. Estimating a Bill
Providers bill per token, usually with separate input/output rates. Given a hypothetical rate, estimate the cost of a request.

In [ ]:
# hypothetical pricing for this exercise — check your provider's actual published rates
INPUT_RATE_PER_1K = 0.05   # USD per 1,000 input tokens
OUTPUT_RATE_PER_1K = 0.08  # USD per 1,000 output tokens

def estimate_cost(input_tokens, output_tokens):
    return (input_tokens / 1000 * INPUT_RATE_PER_1K) + (output_tokens / 1000 * OUTPUT_RATE_PER_1K)

sample_input_tokens = len(enc.encode(prompt))
sample_output_tokens = 80  # matches max_tokens above

cost = estimate_cost(sample_input_tokens, sample_output_tokens)
print(f"input tokens: {sample_input_tokens}, output tokens (max): {sample_output_tokens}")
print(f"estimated cost: ${cost:.5f}")

### Week 2, Class 2 — closed
Every call you make to an LLM API is really three numbers underneath: how many tokens went in, how many came out, and how randomly they were sampled. You now know how to measure and control all three.

## Challenges
Work through these in order. No solutions are provided — each starter cell has a `# TODO` marking where your code goes. Challenge 2 needs your own `GROQ_API_KEY`.

### Challenge 1 — Token-Count a Paragraph
Write your own 2-3 sentence paragraph. Guess how many tokens it will be, then encode it with `enc.encode` and print the actual count next to your guess.

**Acceptance criteria:** prints your guess and the real token count for the same paragraph.

In [ ]:
# TODO: write a paragraph, guess its token count, then print your guess vs. enc.encode(paragraph)

### Challenge 2 — Sweep Temperature
Call `call_llm` with the same prompt at `temperature` 0, 0.7, and 1.5. Print all three outputs and, in a comment, describe how they differ.

**Acceptance criteria:** prints three completions at three different temperatures, plus a one-line comment describing the difference.

In [ ]:
# TODO: call call_llm(prompt, temperature=...) at 0, 0.7, and 1.5; print all three plus a comment

### Challenge 3 — Estimate a 5,000-Token Request
Using `estimate_cost` from Section 3, compute the cost of a request with 4,000 input tokens and 1,000 output tokens.

**Acceptance criteria:** prints a single dollar-amount cost for 4,000 input / 1,000 output tokens.

In [ ]:
# TODO: call estimate_cost(4000, 1000) and print the result

### Challenge 4 — Simulate Context-Window Truncation
Given a list of chat turns (strings) and a small pretend token budget, write `fit_to_budget(turns, budget)` that drops the *oldest* turns (from the front of the list) until the total token count (via `enc.encode`) fits within budget. Test it on the 6 turns below with `budget=50`.

**Acceptance criteria:** returns a list that is a suffix of `turns`, and the total token count of the returned turns is at most `budget`.

In [ ]:
turns = [
    "Hi, I need help with my order.",
    "Sure, what's your order number?",
    "It's 48213.",
    "Let me look that up for you.",
    "It shipped yesterday and should arrive Friday.",
    "Great, thank you!",
]

# TODO: write fit_to_budget(turns, budget) using enc.encode to count tokens per turn,
# dropping oldest turns until the total fits, then call fit_to_budget(turns, 50)